In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

# CIFAR-10 — CNN Classification

We compare two architecturally distinct approaches:
- **Architecture 1 — Custom CNN**: designed from scratch with 3 convolutional blocks
- **Architecture 2 — VGG19 Transfer Learning**: pre-trained ImageNet weights, fine-tuned on CIFAR-10

The custom CNN is tested with and without regularisation.

### Imports

In [ ]:
import gc
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
%matplotlib inline

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras import backend as K
from tensorflow.keras.applications.vgg19 import VGG19
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.layers import (Input, Conv2D, Dense, Dropout,
                                      Flatten, MaxPool2D, BatchNormalization)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.random import set_seed

print("Tensorflow version " + tf.__version__)

### Data loading & preprocessing

In [ ]:
batch_size = 128
classes    = 10

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

(X_train, y_train), (X_test, y_test) = cifar10.load_data()

input_shape = (32, 32, 3)
X_train = X_train.astype('float32') / 255
X_test  = X_test.astype('float32')  / 255

Y_train = to_categorical(y_train, classes)
Y_test  = to_categorical(y_test,  classes)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

### Visualise samples

In [ ]:
plt.style.use('dark_background')
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i])
    ax.set_title(class_names[y_train[i][0]], fontsize=9)
    ax.axis('off')
plt.suptitle('CIFAR-10 — sample images', fontsize=13)
plt.tight_layout()
plt.show()

### Helper functions

In [ ]:
def plot_history(hs, total_epochs, metric, max_y=1.0):
    """
    hs: dict of {label: history_object}  OR
        dict of {label: {'history': hist, 'x_start': int}}
    """
    plt.style.use('dark_background')
    plt.rcParams['figure.figsize'] = [15, 8]
    plt.rcParams['font.size'] = 16
    plt.clf()
    for label, entry in hs.items():
        if isinstance(entry, dict):
            hist    = entry['history']
            x_start = entry.get('x_start', 1)
        else:
            hist    = entry
            x_start = 1
        vals     = hist.history[metric]
        val_vals = hist.history[f'val_{metric}']
        xs = np.arange(x_start, x_start + len(vals))
        plt.plot(xs, vals,     label=f'{label} train {metric}',  linewidth=2)
        plt.plot(xs, val_vals, label=f'{label} val {metric}',    linewidth=2)
    x_ticks = np.arange(0, total_epochs + 1, max(1, total_epochs // 10))
    x_ticks[0] += 1
    plt.xticks(x_ticks)
    plt.ylim((0, max_y))
    plt.xlabel('Epochs')
    plt.ylabel('Loss' if metric == 'loss' else 'Accuracy')
    plt.legend()
    plt.show()


def plot_confusion_matrix(model, title='Confusion Matrix'):
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    cm = confusion_matrix(y_test, y_pred)
    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=True, xticks_rotation=45)
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    plt.show()


def clean_up(model):
    K.clear_session()
    del model
    gc.collect()

---
## Architecture 1 — Custom CNN

```
Input(32,32,3)
  → Conv2D(32,  3×3, ReLU) → [BN] → MaxPool(2×2) → [Dropout(0.2)]
  → Conv2D(64,  3×3, ReLU) → [BN] → MaxPool(2×2) → [Dropout(0.2)]
  → Conv2D(128, 3×3, ReLU) → [BN] → MaxPool(2×2) → [Dropout(0.2)]
  → Flatten → Dense(256, ReLU) → [Dropout(0.3)] → Output(10, softmax)
```

Experiments:
1. **No regularisation** — baseline
2. **BatchNorm + Dropout + Early Stopping** — regularised

In [ ]:
def build_custom_cnn(dropout=False, batchnorm=False):
    np.random.seed(1402)
    set_seed(1981)

    inp = Input(shape=input_shape, name='Input')
    x = inp
    for i, filters in enumerate([32, 64, 128]):
        x = Conv2D(filters, (3,3), padding='same',
                   activation='relu', name=f'Conv2D-{i+1}')(x)
        if batchnorm:
            x = BatchNormalization(name=f'BN-{i+1}')(x)
        x = MaxPool2D((2,2), name=f'MaxPool-{i+1}')(x)
        if dropout:
            x = Dropout(0.2, name=f'Drop-{i+1}')(x)

    x = Flatten(name='Flatten')(x)
    x = Dense(256, activation='relu', name='FC-1')(x)
    if dropout:
        x = Dropout(0.3, name='Drop-FC')(x)
    out = Dense(classes, activation='softmax', name='Output')(x)
    return Model(inputs=inp, outputs=out)


def train(model, optimizer, epochs, batch_size, callbacks=None, verbose=0):
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy', metrics=['accuracy'])
    hs = model.fit(X_train, Y_train, validation_split=0.1,
                   epochs=epochs, batch_size=batch_size,
                   callbacks=callbacks, verbose=verbose)
    print('Finished training.')
    print('------------------')
    model.summary()
    return hs

### Custom CNN — Experiment 1a: No regularisation

In [ ]:
cnn_base = build_custom_cnn(dropout=False, batchnorm=False)
cnn_base_hs = train(cnn_base, Adam(), 100, batch_size)
cnn_base_eval = cnn_base.evaluate(X_test, Y_test, verbose=1)

print(f"\nTest Acc: {cnn_base_eval[1]:.5f}  Test Loss: {cnn_base_eval[0]:.5f}")
plot_history({'Custom CNN (base)': cnn_base_hs}, 100, 'loss')
plot_history({'Custom CNN (base)': cnn_base_hs}, 100, 'accuracy')

clean_up(cnn_base)

### Custom CNN — Experiment 1b: BatchNorm + Dropout + Early Stopping

In [ ]:
es = EarlyStopping(monitor='val_accuracy', patience=10,
                   verbose=1, restore_best_weights=True)

cnn_reg = build_custom_cnn(dropout=True, batchnorm=True)
cnn_reg_hs = train(cnn_reg, Adam(), 100, batch_size,
                   callbacks=[es], verbose=1)
cnn_reg_eval = cnn_reg.evaluate(X_test, Y_test, verbose=1)

ep = len(cnn_reg_hs.history['loss'])
print(f"\nStopped @ epoch {ep}")
print(f"Test Acc: {cnn_reg_eval[1]:.5f}  Test Loss: {cnn_reg_eval[0]:.5f}")
plot_history({'Custom CNN (base)':      cnn_base_hs,
              'Custom CNN (BN+Drop+ES)': cnn_reg_hs}, 100, 'loss')
plot_history({'Custom CNN (base)':      cnn_base_hs,
              'Custom CNN (BN+Drop+ES)': cnn_reg_hs}, 100, 'accuracy')

### Confusion matrix — Custom CNN (regularised)

In [ ]:
plot_confusion_matrix(cnn_reg, title='Confusion Matrix — Custom CNN (BN + Dropout)')
clean_up(cnn_reg)

---
## Architecture 2 — VGG19 Transfer Learning

VGG19 pre-trained on ImageNet (1000-class, 224×224 images) is used as a feature extractor.
Since CIFAR-10 images are 32×32, the model receives them at low resolution — the pre-trained
spatial features still provide a strong initialisation.

**Two-phase training:**
1. **Phase 1 (30 epochs)** — freeze VGG19 base, train only the new dense head with Adam (default lr)
2. **Phase 2 (up to 70 epochs)** — unfreeze all layers, fine-tune with Adam (lr=1e-5) + Early Stopping

In [ ]:
def train_vgg19(
        upper_optimizer,
        full_optimizer,
        upper_epochs=30,
        full_epochs=70,
        batch_size=128,
        callbacks=None,
        verbose=0):

    np.random.seed(1402)
    set_seed(1981)

    base = VGG19(include_top=False, weights='imagenet',
                 input_shape=input_shape, pooling='max')

    x = base.output
    x = Dense(512, activation='relu',  name='Hidden-1')(x)
    x = Dropout(0.2,                   name='Drop-1')(x)
    x = Dense(256, activation='relu',  name='Hidden-2')(x)
    x = Dropout(0.2,                   name='Drop-2')(x)
    out = Dense(classes, activation='softmax', name='Output')(x)
    model = Model(inputs=base.input, outputs=out)

    # Phase 1 — frozen base
    for layer in base.layers:
        layer.trainable = False
    model.compile(optimizer=upper_optimizer,
                  loss='categorical_crossentropy', metrics=['accuracy'])
    hs_upper = model.fit(X_train, Y_train, validation_split=0.1,
                         epochs=upper_epochs, batch_size=batch_size,
                         callbacks=callbacks, verbose=verbose)
    print(f'Phase 1 done ({len(hs_upper.history["loss"])} epochs).')

    # Phase 2 — full fine-tuning
    for layer in base.layers:
        layer.trainable = True
    model.compile(optimizer=full_optimizer,
                  loss='categorical_crossentropy', metrics=['accuracy'])
    hs_full = model.fit(X_train, Y_train, validation_split=0.1,
                        epochs=full_epochs, batch_size=batch_size,
                        callbacks=callbacks, verbose=verbose)
    print(f'Phase 2 done ({len(hs_full.history["loss"])} epochs).')
    model.summary()
    return model, hs_upper, hs_full

### VGG19 — train and evaluate

In [ ]:
es = EarlyStopping(monitor='val_accuracy', patience=10,
                   verbose=1, restore_best_weights=True)

vgg_model, vgg_upper_hs, vgg_full_hs = train_vgg19(
    upper_optimizer=Adam(),
    full_optimizer=Adam(learning_rate=1e-5),
    upper_epochs=30,
    full_epochs=70,
    batch_size=batch_size,
    callbacks=[es],
    verbose=1
)
vgg_eval = vgg_model.evaluate(X_test, Y_test, verbose=1)

In [ ]:
upper_ep = len(vgg_upper_hs.history['loss'])
full_ep  = len(vgg_full_hs.history['loss'])
total_ep = upper_ep + full_ep

print('=== VGG19 Transfer Learning ===')
print(f'Phase 1  Train Acc: {vgg_upper_hs.history["accuracy"][-1]:.5f}  '
      f'Val Acc: {vgg_upper_hs.history["val_accuracy"][-1]:.5f}')
print(f'Phase 2  Train Acc: {vgg_full_hs.history["accuracy"][-1]:.5f}  '
      f'Val Acc: {vgg_full_hs.history["val_accuracy"][-1]:.5f}')
print(f'Test Loss: {vgg_eval[0]:.5f}  Test Acc: {vgg_eval[1]:.5f}')

plot_history({
    'Phase 1 (frozen)':   {'history': vgg_upper_hs, 'x_start': 1},
    'Phase 2 (finetune)': {'history': vgg_full_hs,  'x_start': upper_ep + 1}
}, total_ep, 'loss')
plot_history({
    'Phase 1 (frozen)':   {'history': vgg_upper_hs, 'x_start': 1},
    'Phase 2 (finetune)': {'history': vgg_full_hs,  'x_start': upper_ep + 1}
}, total_ep, 'accuracy')

### Confusion matrix — VGG19

In [ ]:
plot_confusion_matrix(vgg_model, title='Confusion Matrix — VGG19 Transfer Learning')
clean_up(vgg_model)

---
## Final comparison — all models on CIFAR-10

In [ ]:
print(f"{'Model':<38} {'Test Acc':>10} {'Test Loss':>12}")
print('-' * 62)
rows = [
    ('Custom CNN (no regularisation)',  cnn_base_eval),
    ('Custom CNN (BN + Drop + ES)',     cnn_reg_eval),
    ('VGG19 Transfer Learning',         vgg_eval),
]
for name, ev in rows:
    print(f"{name:<38} {ev[1]:>10.5f} {ev[0]:>12.5f}")